# RAG with Reranking — Evaluation (enrichment ablations + optional query rewrite)

This notebook compares retrieval orderings on the same evaluation queries:

1. **Retrieve-only** (no reranking)
2. **Rerank (plain)** — cross-encoder on abstract only
3. **Rerank (enriched, struct only)** — RCSB title + keywords in reranker text; UniProt and chemical blocks **off** (`enrich_uniprot=False`, `enrich_chemical=False`)
4. **Rerank (enriched + UniProt)** — add UniProt function text to the reranker input
5. **Rerank (enriched + chemical)** — add chemical component names to the reranker input
6. **Rerank (enriched + UniProt + chemical)** — both add-ons

Optional (commented in code cells): LLM query rewrite before retrieval, and enriched rerank → **LLM reason → LLM judge** via `run_retrieve_rerank_reason_judge`. Metrics (recall@k, precision@k, MRR, NDCG@k) use the final PMID list for each condition.

**Runtime:** Six rerank passes over all queries dominate cost. Optional rewrite caches to `rag_eval_rewritten_questions.json`. Reason/judge adds roughly N × (2 × `EVAL_RERANK_TOP_K`) LLM forwards per query (batched with `RAD_RAG_LLM_BATCH_SIZE`).

---

**This notebook copy** uses `rag_app.pipeline_seperate_pdb_id` and `rag_app.config_seperate_pdb_id`: reranking scores **one cross-encoder input per (PMID, PDB ID)** (abstract + that PDB’s structure / UniProt / chemical), then aggregates per PMID (`RAD_RAG_PDB_AGGREGATION`, default `max`). Per-query rerank cost can be **higher** than the baseline notebook when PMIDs have many PDB IDs.


In [ ]:
# Set retrieval mode flags: chunk concatenation, abbreviation rewrite, dense-only, or sparse-only.
concat_chunk = False  # Concat knowledge with abstract chunk in the retrieval stage
abbr_rewrite_mode = False # 1. Whether to use the query rewritten module to overcome the abbreviation issue in the question; 2. Load questions with abbr.
abbr_baseline = False # Load questions with abbr.

# Whether to use dense- or sparse-only in retrieval (baseline)
dense_only = False
sparse_only = False



In [ ]:
# Configure project paths and add rag_app to sys.path for importing pipeline modules.
# Environment and path: allow importing rag_app when run from RAD_protein
import sys
import os
from pathlib import Path

rcsb_data_path = "full_rcsb_data.json"
PROJECT_ROOT = Path.cwd()  # repository root (the notebooks live here)
# Inputs: example_data/ ships with the repo; point DATA_DIR at your own data folder for full-scale runs.
DATA_DIR = PROJECT_ROOT / "example_data"
ANNOTATION_DIR = DATA_DIR / "Official_annotation"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Use dedicated eval Chroma dir so we don't touch the UI's chroma_db (avoids corrupted HNSW index)
os.environ["RAD_RAG_DATA_PATH"] = str(DATA_DIR / "demo_pubmed.json")
os.environ["RAD_RAG_CHROMA_DIR"] = str(PROJECT_ROOT / "chroma_db_eval")
os.environ["RAD_RAG_RCSB_DATA_PATH"] = str(DATA_DIR / rcsb_data_path)
os.environ["RAD_RAG_ABBR_DICT_PATH"] = str(ANNOTATION_DIR / "abbr_dict_example.csv")  # full data: abbr_dict_all_abs.csv

# Per-PDB rerank variant (see rag_app.config_seperate_pdb_id)
os.environ.setdefault("RAD_RAG_PDB_AGGREGATION", "max")  # max | mean
os.environ.setdefault("RAD_RAG_INCLUDE_MISSING_PDB_AS_ABSTRACT_ONLY", "0")  # 1 = abstract-only pair if PDB has no RCSB row

if concat_chunk:
    # concat to chunk
    os.environ.setdefault("RAD_RAG_INDEX_CONCAT_METADATA", "1")   # master: 1 is concat
    os.environ.setdefault("RAD_RAG_INDEX_CONCAT_STRUCTURE", "1")
    os.environ.setdefault("RAD_RAG_INDEX_CONCAT_UNIPROT", "0")
    os.environ.setdefault("RAD_RAG_INDEX_CONCAT_CHEMICAL", "0")
else:
    os.environ.setdefault("RAD_RAG_INDEX_CONCAT_METADATA", "0")   # master: 1 is concat
    os.environ.setdefault("RAD_RAG_INDEX_CONCAT_STRUCTURE", "0")
    os.environ.setdefault("RAD_RAG_INDEX_CONCAT_UNIPROT", "0")
    os.environ.setdefault("RAD_RAG_INDEX_CONCAT_CHEMICAL", "0")

import os

# large: 0.6b bge
os.environ["RAD_RAG_EMBED_MODEL"] = 'BAAI/bge-m3'#"BAAI/bge-small-en-v1.5"#BAAI/bge-large-en-v1.5" # "BAAI/bge-small-en-v1.5"
os.environ["RAD_RAG_RERANKER_MODEL"] = "BAAI/bge-reranker-large"#"BAAI/bge-reranker-large"#"BAAI/bge-reranker-v2-m3" #"BAAI/bge-reranker-large"

# small/fast option (roughly analogous to bge-small + strong reranker)
# os.environ["RAD_RAG_EMBED_MODEL"] = "Qwen/Qwen3-Embedding-0.6B"
# os.environ["RAD_RAG_RERANKER_MODEL"] = "Qwen/Qwen3-Reranker-0.6B"

# decomposition
os.environ["RAD_RAG_QUERY_DECOMPOSITION"] = "0"
os.environ["RAD_RAG_QUERY_DECOMPOSITION_RRF_K"] = "10"  # optional, default 60

In [ ]:
# Import retrieval, reranking, and pipeline modules for the evaluation run.
# Imports
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm

# LangChain / retrieval
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.vectorstores import Chroma

# Pipeline: per-PDB rerank variant (re-exports shared helpers from rag_app.pipeline)
from utils.rag_app.pipeline_seperate_pdb_id import (
    load_documents,
    build_splits,
    build_vectorstore,
    build_retriever,
    rerank_entries,
    build_reranker,
    build_llm,
    build_reason_chain,
    build_judge_chain,
    build_query_rewrite_chain,
    rewrite_query,
    run_retrieve_rerank_reason_judge,
    load_pmid_to_abstract,
    load_pmid_to_pdb_ids,
    load_pmid_to_rcsb_metadata,
    maybe_enrich_splits_for_indexing,
    build_query_decomposition_chain,
    decompose_query,
    aggregate_pmids_rrf,
    run_retrieve_rerank,
    run_retrieve_then_rerank_with_other_query,  # NEW
)
from utils.rag_app.abbreviation import (
    load_abbreviation_dict,
    build_abbreviation_regex,
    expand_abbreviations,
)
from utils.rag_app.config_seperate_pdb_id import ABBR_DICT_PATH

In [ ]:
# Print the current embedding model environment variable.
os.environ["RAD_RAG_EMBED_MODEL"]

## Load evaluation data

Build `dict_pmcid_abs`, a list of `(PubMed_ID, question)` pairs, from the question CSV.

In [ ]:
# Load the expanded annotation CSV and rebuild the PubMed ID → question dict from it.
import pandas as pd

if abbr_rewrite_mode:
    path = ANNOTATION_DIR / "single_expand_example_with_abbr_variants.csv"
else:
    if abbr_baseline:
        path = ANNOTATION_DIR / "single_expand_example_with_abbr_variants.csv"
    else:
        path = ANNOTATION_DIR / "single_expand_example.csv"
print(path)
df = pd.read_csv(path)

# Use string IDs as keys
if abbr_baseline or abbr_rewrite_mode:
    dict_pmcid_abs_list = list((zip(df["pmcid"].astype(str), df["with_abbreviation"])))#df["question"])))
else:
    dict_pmcid_abs_list = list((zip(df["pmcid"].astype(str), df["question"])))
    

dict_pmcid_abs = []
for k, v in dict_pmcid_abs_list:
    v = 'Can you'+v.split('Can you')[-1]
    dict_pmcid_abs.append((k, v))


In [ ]:
# Display the current PubMed ID → question dict after loading the CSV.
dict_pmcid_abs

In [ ]:
# Apply abbreviation expansion to each question if abbr_rewrite_mode is enabled.
if abbr_rewrite_mode:
    abbr_dict = load_abbreviation_dict(ABBR_DICT_PATH)
    abbr_regex = build_abbreviation_regex(abbr_dict)
    print(f"Loaded {len(abbr_dict)} unambiguous abbreviations from {ABBR_DICT_PATH}")
    dict_pmcid_abs_raw = list(dict_pmcid_abs)
    expansion_log = []
    for i, (pid, q) in enumerate(dict_pmcid_abs):
        new_q, repl = expand_abbreviations(q, abbr_dict, abbr_regex)
        if repl:
            dict_pmcid_abs[i] = (pid, new_q)
            expansion_log.append((pid, q, new_q, repl))
    print(f"Expanded abbreviations in {len(expansion_log)} / {len(dict_pmcid_abs)} questions.")
    for pid, old, new, repl in expansion_log[:5]:
        print(f"  PMID {pid}")
        print(f"    raw : {old.strip()}")
        print(f"    new : {new.strip()}")
        print(f"    repl: {[(r['abbr'], r['full_name']) for r in repl]}")


In [ ]:
# Print the total number of evaluation queries.
len(dict_pmcid_abs)

In [ ]:
# (Debug) Strip questions to start from 'Can you' prefix.
# dict_pmcid_abs = [(k,'Can you'+v.split('Can you')[-1]) for k,v in dict_pmcid_abs.items()]

In [ ]:
# Print the full question dict and its length.
print(dict_pmcid_abs)
print(len(dict_pmcid_abs))

In [ ]:
# Display the PubMed ID → question dict.
dict_pmcid_abs

## Build RAG + reranker + LLM chains

Load documents, build vectorstore and retriever (`retrieval_k=50`), PMID lookups, cross-encoder, and the same **Qwen** reason/judge chains as the UI. Use `EVAL_RERANK_TOP_K` (e.g. 50) so recall@50 is well-defined for all four conditions.

In [ ]:
# Load documents, build text splits, and construct/load the Chroma vectorstore.
# Documents and splits (same config as the pipeline)
docs = load_documents(str(DATA_DIR / "demo_pubmed.json"))
splits = build_splits(docs, chunk_size=450, chunk_overlap=20) # 450, 20
print(f"Documents: {len(docs)}, Chunks: {len(splits)}")

if concat_chunk:
    pmid_to_rcsb = load_pmid_to_rcsb_metadata()
    # pmid_to_rcsb should already be loaded in your notebook
    splits_for_index = maybe_enrich_splits_for_indexing(splits, pmid_to_rcsb)

    
# Vectorstore: always use a clean Chroma DB to avoid HNSW corruption (load can succeed but query then fails)
import shutil
from utils.rag_app.config_seperate_pdb_id import CHROMA_DIR

eval_chroma_dir = Path(CHROMA_DIR)
if eval_chroma_dir.exists():
    shutil.rmtree(eval_chroma_dir)

if concat_chunk:
    vectorstore = build_vectorstore(splits_for_index, persist=True)
else:
    vectorstore = build_vectorstore(splits, persist=True)

# Retriever and lookups are built in the next cell (with EVAL_RERANK_TOP_K).


In [ ]:
# Print the resolved path to the Chroma vector index directory.
print(CHROMA_DIR.resolve())

In [ ]:
# Inspect the first text split.
splits[0]

In [ ]:
# Inspect the first raw document.
docs[0]

In [ ]:
# Set retrieval K and rerank top-K, then build the hybrid/dense/sparse retriever.
# Config: retrieval k and rerank top-k for evaluation (keep list long enough for recall@50)
RETRIEVAL_K = 100
EVAL_RERANK_TOP_K = 50  # keep top-k after reranking for recall@k eval (k in [1, 5, 10, 20, 50])

splits_used = splits_for_index if concat_chunk else splits

if dense_only:
    print('Dense only')
    retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})
elif sparse_only:
    print('Spare only')
    retriever = BM25Retriever.from_documents(splits_used)
    retriever.k = RETRIEVAL_K
else:
    print('Hybrid')
    if concat_chunk:
        retriever = build_retriever(vectorstore, splits_for_index, k=RETRIEVAL_K)
    else:
        retriever = build_retriever(vectorstore, splits, k=RETRIEVAL_K)

# Lookups for full abstract, PDB IDs, and RCSB metadata per PMID (needed for rerank_entries)
pmid_to_abstract = load_pmid_to_abstract(str(DATA_DIR / "demo_pubmed.json"))
pmid_to_pdb = load_pmid_to_pdb_ids(str(DATA_DIR / rcsb_data_path))
pmid_to_rcsb = load_pmid_to_rcsb_metadata(str(DATA_DIR / rcsb_data_path))

build_reranker()
print("Loading LLM for reason/judge and query rewrite...")
llm = build_llm()
reason_chain = build_reason_chain(llm)
judge_chain = build_judge_chain(llm)
query_rewrite_chain = build_query_rewrite_chain(llm)
query_decomposition_chain = build_query_decomposition_chain(llm)
print("RAG + reranker + LLM chains ready (including query rewrite and decomposition).")

In [ ]:
# (Empty placeholder cell.)


## Single-query pipeline: retrieve → rerank (top-k for eval)

`run_retrieve_rerank` runs retrieve → dedupe → rerank and returns the top-k PMIDs (conditions 2–3). Condition (4) uses `run_retrieve_rerank_reason_judge` from `rag_app.pipeline_seperate_pdb_id` (same ordering logic as the Gradio UI, with chunked LLM batching).

## Evaluation on result lists

For each query, build PMID orderings: retrieve-only; rerank plain; rerank enriched; **rerank enriched with LLM-rewritten query**; optionally enriched rerank + LLM reason/judge reorder. Then compute Recall@k, Precision@k, MRR, MAP, and NDCG@k on each list.

In [ ]:
# Set K_LIST and extract ground-truth PMIDs from the evaluation dict.
# only run this if rewrite
# dict_pmcid_abs = [(k,'Can you'+v.split('Can you')[-1]) for k,v in dict_pmcid_abs.items()]

K_LIST = [1, 5, 10, 20, 50]#, 100]
ground_truth_pmids = [str(pmc_id) for (pmc_id, q) in dict_pmcid_abs]


In [ ]:
# Run all retrieval conditions (retrieve-only through fully enriched rerank) and collect PMID lists.

# (1) Baseline: Retrieve only (no reranking)
retrieval_only_pmid_lists = []
for pmc_id, question_content in tqdm(dict_pmcid_abs, desc="Retrieve only"):
    docs_retrieved = retriever.get_relevant_documents(question_content)
    pmids = []
    for doc in docs_retrieved:
        pmid = str(doc.metadata.get("PubMed_ID", "?"))
        pmids.append(pmid)
    retrieval_only_pmid_lists.append(pmids)

# (2) Retrieval + rerank (plain abstract; no RCSB snippet)
reranked_plain_pmid_lists = []
for pmc_id, question_content in tqdm(dict_pmcid_abs, desc="Rerank (plain)"):
    pmids_reranked = run_retrieve_rerank(
        question_content, retriever, pmid_to_abstract, pmid_to_pdb,
        top_k=EVAL_RERANK_TOP_K, pmid_to_rcsb=None,
    )
    reranked_plain_pmid_lists.append(pmids_reranked)

# (3) Rerank with RCSB structure fields only (title + keywords; UniProt/chem blocks off)
reranked_enriched_pmid_lists = []
for pmc_id, question_content in tqdm(dict_pmcid_abs, desc="Rerank (enriched, struct only)"):
    pmids_reranked = run_retrieve_rerank(
        question_content, retriever, pmid_to_abstract, pmid_to_pdb,
        top_k=EVAL_RERANK_TOP_K, pmid_to_rcsb=pmid_to_rcsb,
        enrich_structure = True,
        enrich_uniprot=False,
        enrich_chemical=False,
    )
    reranked_enriched_pmid_lists.append(pmids_reranked)

# (4) Structure snippet + UniProt function text in cross-encoder input
reranked_enriched_uniprot_pmid_lists = []
for pmc_id, question_content in tqdm(dict_pmcid_abs, desc="Rerank (enriched + UniProt)"):
    pmids_reranked = run_retrieve_rerank(
        question_content, retriever, pmid_to_abstract, pmid_to_pdb,
        top_k=EVAL_RERANK_TOP_K, pmid_to_rcsb=pmid_to_rcsb,
        enrich_structure = True,
        enrich_uniprot=True,
        enrich_chemical=False,
    )
    reranked_enriched_uniprot_pmid_lists.append(pmids_reranked)

# (5) Structure snippet + chemical component names in cross-encoder input
reranked_enriched_chem_pmid_lists = []
for pmc_id, question_content in tqdm(dict_pmcid_abs, desc="Rerank (enriched + chemical)"):
    pmids_reranked = run_retrieve_rerank(
        question_content, retriever, pmid_to_abstract, pmid_to_pdb,
        top_k=EVAL_RERANK_TOP_K, pmid_to_rcsb=pmid_to_rcsb,
        enrich_structure = True,
        enrich_uniprot=False,
        enrich_chemical=True,
    )
    reranked_enriched_chem_pmid_lists.append(pmids_reranked)


# (6) Structure + UniProt + chemical components (full metadata text)
reranked_enriched_both_pmid_lists = []
for pmc_id, question_content in tqdm(dict_pmcid_abs, desc="Rerank (enriched + UniProt + chem)"):
    pmids_reranked = run_retrieve_rerank(
        question_content, retriever, pmid_to_abstract, pmid_to_pdb,
        top_k=EVAL_RERANK_TOP_K, pmid_to_rcsb=pmid_to_rcsb,
        enrich_structure = True,
        enrich_uniprot=True,
        enrich_chemical=True,
    )
    reranked_enriched_both_pmid_lists.append(pmids_reranked)



# # (8) Enriched rerank → reason → judge (uncomment if chains are available)
# reranked_enriched_rj_pmid_lists = []
# for pmc_id, question_content in tqdm(dict_pmcid_abs, desc="Rerank (enriched) + R/J"):
#     pmids_ordered, _entries = run_retrieve_rerank_reason_judge(
#         question_content,
#         retriever,
#         pmid_to_abstract,
#         pmid_to_pdb,
#         reason_chain,
#         judge_chain,
#         top_k=EVAL_RERANK_TOP_K,
#         pmid_to_rcsb=pmid_to_rcsb,
#         pmid_to_fulltext=None,
#         progress_callback=None,
#         query_rewrite_chain=query_rewrite_chain,
#         enrich_uniprot=True,
#         enrich_chemical=True,
#     )
#     reranked_enriched_rj_pmid_lists.append(pmids_ordered)

# Backward compat for later cells that expect reranked_pmid_lists / results_rerank
'''
reranked_pmid_lists = reranked_enriched_pmid_lists
'''

In [ ]:
# (Debug) Sixth retrieval condition: rerank with structure + UniProt + chemical enrichment.
# # (6) Structure + UniProt + chemical components (full metadata text)
# reranked_enriched_both_pmid_lists = []
# for pmc_id, question_content in tqdm(dict_pmcid_abs, desc="Rerank (enriched + UniProt + chem)"):
#     pmids_reranked = run_retrieve_rerank(
#         question_content, retriever, pmid_to_abstract, pmid_to_pdb,
#         top_k=EVAL_RERANK_TOP_K, pmid_to_rcsb=pmid_to_rcsb,
#         enrich_structure = True,
#         enrich_uniprot=True,
#         enrich_chemical=True,
#     )
#     reranked_enriched_both_pmid_lists.append(pmids_reranked)

# reranked_enriched_both_pmid_lists = reranked_enriched_chem_pmid_lists

In [ ]:
# Alias the best rerank list and compute Recall@K for all conditions.
# Backward compat for later cells that expect reranked_pmid_lists / results_rerank
reranked_pmid_lists = reranked_enriched_pmid_lists

def compute_recall_results(pmid_lists, label):
    res = {}
    for k in K_LIST:
        recall_flags = [
            1 if gt in (pmids[:k]) else 0
            for pmids, gt in zip(pmid_lists, ground_truth_pmids)
        ]
        recall_flags = np.array(recall_flags)
        res[k] = (
            recall_flags.mean() * 100,
            recall_flags.std(ddof=1) * 100 if len(recall_flags) > 1 else 0.0,
        )
        print(f"{label} recall@{k}: {res[k][0]:.2f} ({res[k][1]:.2f})")
    return res

results_baseline = compute_recall_results(retrieval_only_pmid_lists, "Retrieve-only")
print("")
results_rerank_plain = compute_recall_results(reranked_plain_pmid_lists, "Rerank (plain)")
print("")
results_rerank_enriched = compute_recall_results(
    reranked_enriched_pmid_lists, "Rerank (enriched, struct only)"
)
print("")


results_rerank_enriched_uniprot = compute_recall_results(
    reranked_enriched_uniprot_pmid_lists, "Rerank (enriched, structure + UniProt)"
)
print("")
results_rerank_enriched_chem = compute_recall_results(
    reranked_enriched_chem_pmid_lists, "Rerank (enriched, structure + chemical)"
)
print("")
results_rerank_enriched_both = compute_recall_results(
    reranked_enriched_both_pmid_lists, "Rerank (enriched, structure + UniProt + chemical)"
)
print("")


results_rerank = results_rerank_enriched  # main config (struct-only enriched rerank)
print("\nSummary (recall %, mean over queries):")
print("k\t" + "\t".join(str(k) for k in K_LIST))
print("Retrieve-only\t" + "\t".join([f"{results_baseline[k][0]:.2f}" for k in K_LIST]))
print("Rerank (plain)\t" + "\t".join([f"{results_rerank_plain[k][0]:.2f}" for k in K_LIST]))
print("Rerank (enriched, struct)\t" + "\t".join([f"{results_rerank_enriched[k][0]:.2f}" for k in K_LIST]))
# print("Rerank (enriched, rewrite, plain)\t" + "\t".join([f"{results_rerank_rewrite[k][0]:.2f}" for k in K_LIST]))
# print("Rerank (enriched, rewrite, struct)\t" + "\t".join([f"{results_rerank_enriched_rewrite[k][0]:.2f}" for k in K_LIST]))
print("Rerank (struct+UniProt)\t" + "\t".join([f"{results_rerank_enriched_uniprot[k][0]:.2f}" for k in K_LIST]))
print("Rerank (struct+chemical)\t" + "\t".join([f"{results_rerank_enriched_chem[k][0]:.2f}" for k in K_LIST]))
print("Rerank (struct+UniProt+chem)\t" + "\t".join([f"{results_rerank_enriched_both[k][0]:.2f}" for k in K_LIST]))


**Note:** Recall / precision for all six rerank variants are computed in the **previous code cell**. This cell used to duplicate that logic; run the cell above only once per experiment.


In [ ]:
# Compute extended metrics: Precision@K, MRR, MAP, and NDCG@K for all conditions.
# Extended metrics: Precision@k, MRR, MAP, NDCG@k (all active conditions)

def rank_of_gt(pmids, gt):
    """1-based rank of gt in pmids, or None if not found."""
    for i, p in enumerate(pmids):
        if p == gt:
            return i + 1
    return None


def compute_extended_metrics(pmid_lists):
    prec, rr, ndcg = {}, [], {}
    for k in K_LIST:
        precs = [
            (1.0 / k) if gt in (pmids[:k]) else 0.0
            for pmids, gt in zip(pmid_lists, ground_truth_pmids)
        ]
        prec[k] = (
            np.mean(precs) * 100,
            np.std(precs, ddof=1) * 100 if len(precs) > 1 else 0.0,
        )
    for pmids, gt in zip(pmid_lists, ground_truth_pmids):
        r = rank_of_gt(pmids, gt)
        rr.append(1.0 / r if r is not None else 0.0)
    mrr = float(np.mean(rr))
    for k in K_LIST:
        ndcgs = []
        for pmids, gt in zip(pmid_lists, ground_truth_pmids):
            top_k = pmids[:k]
            if gt not in top_k:
                ndcgs.append(0.0)
            else:
                rank = rank_of_gt(top_k, gt)
                ndcgs.append(1.0 / np.log2(rank + 1))
        ndcg[k] = (
            float(np.mean(ndcgs)),
            float(np.std(ndcgs, ddof=1)) if len(ndcgs) > 1 else 0.0,
        )
    return prec, mrr, ndcg


results_precision_baseline, mrr_baseline, results_ndcg_baseline = compute_extended_metrics(
    retrieval_only_pmid_lists
)
results_precision_plain, mrr_plain, results_ndcg_plain = compute_extended_metrics(
    reranked_plain_pmid_lists
)
results_precision_enriched, mrr_enriched, results_ndcg_enriched = compute_extended_metrics(
    reranked_enriched_pmid_lists
)


results_precision_enriched_uniprot, mrr_enriched_uniprot, results_ndcg_enriched_uniprot = (
    compute_extended_metrics(reranked_enriched_uniprot_pmid_lists)
)
results_precision_enriched_chem, mrr_enriched_chem, results_ndcg_enriched_chem = (
    compute_extended_metrics(reranked_enriched_chem_pmid_lists)
)
results_precision_enriched_both, mrr_enriched_both, results_ndcg_enriched_both = (
    compute_extended_metrics(reranked_enriched_both_pmid_lists)
)

map_baseline = mrr_baseline
map_plain = mrr_plain
map_enriched = mrr_enriched

map_enriched_uniprot = mrr_enriched_uniprot
map_enriched_chem = mrr_enriched_chem
map_enriched_both = mrr_enriched_both

print("Extended metrics (Retrieve | Plain | Enriched struct | +UniProt | +chem | +both):")
print(
    "  Retrieve-only: Precision@1",
    f"{results_precision_baseline[1][0]:.2f}",
    "MRR",
    f"{mrr_baseline:.4f}",
    "NDCG@10",
    f"{results_ndcg_baseline[10][0]:.4f}",
)
print(
    "  Rerank (plain): Precision@1",
    f"{results_precision_plain[1][0]:.2f}",
    "MRR",
    f"{mrr_plain:.4f}",
    "NDCG@10",
    f"{results_ndcg_plain[10][0]:.4f}",
)
print(
    "  Rerank (enriched, struct): Precision@1",
    f"{results_precision_enriched[1][0]:.2f}",
    "MRR",
    f"{mrr_enriched:.4f}",
    "NDCG@10",
    f"{results_ndcg_enriched[10][0]:.4f}",
)
print(
    "  Rerank (+UniProt): Precision@1",
    f"{results_precision_enriched_uniprot[1][0]:.2f}",
    "MRR",
    f"{mrr_enriched_uniprot:.4f}",
    "NDCG@10",
    f"{results_ndcg_enriched_uniprot[10][0]:.4f}",
)
print(
    "  Rerank (+chemical): Precision@1",
    f"{results_precision_enriched_chem[1][0]:.2f}",
    "MRR",
    f"{mrr_enriched_chem:.4f}",
    "NDCG@10",
    f"{results_ndcg_enriched_chem[10][0]:.4f}",
)
print(
    "  Rerank (+UniProt+chem): Precision@1",
    f"{results_precision_enriched_both[1][0]:.2f}",
    "MRR",
    f"{mrr_enriched_both:.4f}",
    "NDCG@10",
    f"{results_ndcg_enriched_both[10][0]:.4f}",
)

print("\n--- Summary (active conditions) ---")
print("k\t" + "\t".join(str(k) for k in K_LIST))
print("Recall (Retrieve)\t" + "\t".join([f"{results_baseline[k][0]:.2f}" for k in K_LIST]))
print("Recall (Plain)\t" + "\t".join([f"{results_rerank_plain[k][0]:.2f}" for k in K_LIST]))
print("Recall (Enriched, struct)\t" + "\t".join([f"{results_rerank_enriched[k][0]:.2f}" for k in K_LIST]))
print("Recall (+UniProt)\t" + "\t".join([f"{results_rerank_enriched_uniprot[k][0]:.2f}" for k in K_LIST]))
print("Recall (+chemical)\t" + "\t".join([f"{results_rerank_enriched_chem[k][0]:.2f}" for k in K_LIST]))
print("Recall (+UniProt+chem)\t" + "\t".join([f"{results_rerank_enriched_both[k][0]:.2f}" for k in K_LIST]))
print(
    "Precision (Retrieve)\t"
    + "\t".join([f"{results_precision_baseline[k][0]:.2f}" for k in K_LIST])
)
print("Precision (Plain)\t" + "\t".join([f"{results_precision_plain[k][0]:.2f}" for k in K_LIST]))
print(
    "Precision (Enriched, struct)\t"
    + "\t".join([f"{results_precision_enriched[k][0]:.2f}" for k in K_LIST])
)
print(
    "Precision (+UniProt)\t"
    + "\t".join([f"{results_precision_enriched_uniprot[k][0]:.2f}" for k in K_LIST])
)
print(
    "Precision (+chemical)\t"
    + "\t".join([f"{results_precision_enriched_chem[k][0]:.2f}" for k in K_LIST])
)
print(
    "Precision (+UniProt+chem)\t"
    + "\t".join([f"{results_precision_enriched_both[k][0]:.2f}" for k in K_LIST])
)
print(
    "NDCG (Retrieve)\t"
    + "\t".join([f"{results_ndcg_baseline[k][0]:.4f}" for k in K_LIST])
)
print("NDCG (Plain)\t" + "\t".join([f"{results_ndcg_plain[k][0]:.4f}" for k in K_LIST]))
print(
    "NDCG (Enriched, struct)\t"
    + "\t".join([f"{results_ndcg_enriched[k][0]:.4f}" for k in K_LIST])
)
print(
    "NDCG (+UniProt)\t"
    + "\t".join([f"{results_ndcg_enriched_uniprot[k][0]:.4f}" for k in K_LIST])
)
print(
    "NDCG (+chemical)\t"
    + "\t".join([f"{results_ndcg_enriched_chem[k][0]:.4f}" for k in K_LIST])
)
print(
    "NDCG (+UniProt+chem)\t"
    + "\t".join([f"{results_ndcg_enriched_both[k][0]:.4f}" for k in K_LIST])
)


In [ ]:
# Compute 95% bootstrap confidence intervals for Recall@K, Precision@K, and NDCG@K.
# 95% ci

# Backward compat for later cells that expect reranked_pmid_lists / results_rerank
reranked_pmid_lists = reranked_enriched_pmid_lists

# 95% CI via nonparametric bootstrap over queries
N_BOOT = 10000
BOOT_SEED = 42
CI_LOW_Q, CI_HIGH_Q = 2.5, 97.5


def _ci(values, n_boot=N_BOOT, seed=BOOT_SEED):
    """Bootstrap (mean, ci_low, ci_high) by resampling query indices with replacement."""
    arr = np.asarray(values, dtype=float)
    n = len(arr)
    if n == 0:
        return 0.0, 0.0, 0.0
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = arr[idx].mean(axis=1)
    lo, hi = np.percentile(boot_means, [CI_LOW_Q, CI_HIGH_Q])
    return float(arr.mean()), float(lo), float(hi)


def compute_recall_results_ci(pmid_lists, label):
    res = {}
    for k in K_LIST:
        flags = np.array([
            1.0 if gt in pmids[:k] else 0.0
            for pmids, gt in zip(pmid_lists, ground_truth_pmids)
        ])
        mean, lo, hi = _ci(flags)
        # store as percentages to match your existing format
        res[k] = (mean * 100, lo * 100, hi * 100)
        print(f"{label} recall@{k}: {res[k][0]:.2f} [95% CI: {res[k][1]:.2f}, {res[k][2]:.2f}]")
    return res


results_baseline = compute_recall_results_ci(retrieval_only_pmid_lists, "Retrieve-only")
print("")
results_rerank_plain = compute_recall_results_ci(reranked_plain_pmid_lists, "Rerank (plain)")
print("")
results_rerank_enriched = compute_recall_results_ci(
    reranked_enriched_pmid_lists, "Rerank (enriched, struct only)"
)
print("")
results_rerank_enriched_uniprot = compute_recall_results_ci(
    reranked_enriched_uniprot_pmid_lists, "Rerank (enriched, structure + UniProt)"
)
print("")
results_rerank_enriched_chem = compute_recall_results_ci(
    reranked_enriched_chem_pmid_lists, "Rerank (enriched, structure + chemical)"
)
print("")
results_rerank_enriched_both = compute_recall_results_ci(
    reranked_enriched_both_pmid_lists, "Rerank (enriched, structure + UniProt + chemical)"
)
print("")


results_rerank = results_rerank_enriched  # main config (struct-only enriched rerank)
print("\nSummary (recall %, mean [95% CI low, high] over queries):")
print("k\t" + "\t".join(str(k) for k in K_LIST))


def _fmt_row(label, res):
    cells = [f"{res[k][0]:.2f} [{res[k][1]:.2f}, {res[k][2]:.2f}]" for k in K_LIST]
    return label + "\t" + "\t".join(cells)


print(_fmt_row("Retrieve-only", results_baseline))
print(_fmt_row("Rerank (plain)", results_rerank_plain))
print(_fmt_row("Rerank (enriched, struct)", results_rerank_enriched))
print(_fmt_row("Rerank (struct+UniProt)", results_rerank_enriched_uniprot))
print(_fmt_row("Rerank (struct+chemical)", results_rerank_enriched_chem))
print(_fmt_row("Rerank (struct+UniProt+chem)", results_rerank_enriched_both))


# Extended metrics: Precision@k, MRR, NDCG@k (all active conditions), each with 95% bootstrap CI


def rank_of_gt(pmids, gt):
    """1-based rank of gt in pmids, or None if not found."""
    for i, p in enumerate(pmids):
        if p == gt:
            return i + 1
    return None


def compute_extended_metrics_ci(pmid_lists):
    prec, ndcg = {}, {}

    # Precision@k
    for k in K_LIST:
        precs = np.array([
            (1.0 / k) if gt in pmids[:k] else 0.0
            for pmids, gt in zip(pmid_lists, ground_truth_pmids)
        ])
        mean, lo, hi = _ci(precs)
        prec[k] = (mean * 100, lo * 100, hi * 100)

    # MRR
    rr = []
    for pmids, gt in zip(pmid_lists, ground_truth_pmids):
        r = rank_of_gt(pmids, gt)
        rr.append(1.0 / r if r is not None else 0.0)
    mrr_mean, mrr_lo, mrr_hi = _ci(rr)
    mrr_tuple = (mrr_mean, mrr_lo, mrr_hi)

    # NDCG@k
    for k in K_LIST:
        ndcgs = []
        for pmids, gt in zip(pmid_lists, ground_truth_pmids):
            top_k = pmids[:k]
            if gt not in top_k:
                ndcgs.append(0.0)
            else:
                rank = rank_of_gt(top_k, gt)
                ndcgs.append(1.0 / np.log2(rank + 1))
        mean, lo, hi = _ci(ndcgs)
        ndcg[k] = (mean, lo, hi)

    return prec, mrr_tuple, ndcg


results_precision_baseline, mrr_baseline, results_ndcg_baseline = compute_extended_metrics_ci(
    retrieval_only_pmid_lists
)
results_precision_plain, mrr_plain, results_ndcg_plain = compute_extended_metrics_ci(
    reranked_plain_pmid_lists
)
results_precision_enriched, mrr_enriched, results_ndcg_enriched = compute_extended_metrics_ci(
    reranked_enriched_pmid_lists
)
results_precision_enriched_uniprot, mrr_enriched_uniprot, results_ndcg_enriched_uniprot = (
    compute_extended_metrics_ci(reranked_enriched_uniprot_pmid_lists)
)
results_precision_enriched_chem, mrr_enriched_chem, results_ndcg_enriched_chem = (
    compute_extended_metrics_ci(reranked_enriched_chem_pmid_lists)
)
results_precision_enriched_both, mrr_enriched_both, results_ndcg_enriched_both = (
    compute_extended_metrics_ci(reranked_enriched_both_pmid_lists)
)

# MAP == MRR in single-relevant-doc setting (preserve your existing aliases)
map_baseline = mrr_baseline
map_plain = mrr_plain
map_enriched = mrr_enriched
map_enriched_uniprot = mrr_enriched_uniprot
map_enriched_chem = mrr_enriched_chem
map_enriched_both = mrr_enriched_both


def _fmt_pct(t):
    return f"{t[0]:.2f} [{t[1]:.2f}, {t[2]:.2f}]"


def _fmt_score(t):
    return f"{t[0]:.4f} [{t[1]:.4f}, {t[2]:.4f}]"


print("\n--- Headline numbers (Precision@1, MRR, NDCG@10) ---")
print(
    "Retrieve-only         | P@1",  _fmt_pct(results_precision_baseline[1]),
    "| MRR", _fmt_score(mrr_baseline),
    "| NDCG@10", _fmt_score(results_ndcg_baseline[10]),
)
print(
    "Rerank (plain)        | P@1", _fmt_pct(results_precision_plain[1]),
    "| MRR", _fmt_score(mrr_plain),
    "| NDCG@10", _fmt_score(results_ndcg_plain[10]),
)
print(
    "Rerank (enriched)     | P@1", _fmt_pct(results_precision_enriched[1]),
    "| MRR", _fmt_score(mrr_enriched),
    "| NDCG@10", _fmt_score(results_ndcg_enriched[10]),
)
print(
    "Rerank (+UniProt)     | P@1", _fmt_pct(results_precision_enriched_uniprot[1]),
    "| MRR", _fmt_score(mrr_enriched_uniprot),
    "| NDCG@10", _fmt_score(results_ndcg_enriched_uniprot[10]),
)
print(
    "Rerank (+chemical)    | P@1", _fmt_pct(results_precision_enriched_chem[1]),
    "| MRR", _fmt_score(mrr_enriched_chem),
    "| NDCG@10", _fmt_score(results_ndcg_enriched_chem[10]),
)
print(
    "Rerank (+UniProt+chem)| P@1", _fmt_pct(results_precision_enriched_both[1]),
    "| MRR", _fmt_score(mrr_enriched_both),
    "| NDCG@10", _fmt_score(results_ndcg_enriched_both[10]),
)


print("\n--- Summary (active conditions) ---")
print("k\t" + "\t".join(str(k) for k in K_LIST))


def _fmt_row_pct(label, res, k_list=K_LIST):
    return label + "\t" + "\t".join([_fmt_pct(res[k]) for k in k_list])


def _fmt_row_score(label, res, k_list=K_LIST):
    return label + "\t" + "\t".join([_fmt_score(res[k]) for k in k_list])


print(_fmt_row_pct("Recall (Retrieve)", results_baseline))
print(_fmt_row_pct("Recall (Plain)", results_rerank_plain))
print(_fmt_row_pct("Recall (Enriched, struct)", results_rerank_enriched))
print(_fmt_row_pct("Recall (+UniProt)", results_rerank_enriched_uniprot))
print(_fmt_row_pct("Recall (+chemical)", results_rerank_enriched_chem))
print(_fmt_row_pct("Recall (+UniProt+chem)", results_rerank_enriched_both))

print(_fmt_row_pct("Precision (Retrieve)", results_precision_baseline))
print(_fmt_row_pct("Precision (Plain)", results_precision_plain))
print(_fmt_row_pct("Precision (Enriched, struct)", results_precision_enriched))
print(_fmt_row_pct("Precision (+UniProt)", results_precision_enriched_uniprot))
print(_fmt_row_pct("Precision (+chemical)", results_precision_enriched_chem))
print(_fmt_row_pct("Precision (+UniProt+chem)", results_precision_enriched_both))

print(_fmt_row_score("NDCG (Retrieve)", results_ndcg_baseline))
print(_fmt_row_score("NDCG (Plain)", results_ndcg_plain))
print(_fmt_row_score("NDCG (Enriched, struct)", results_ndcg_enriched))
print(_fmt_row_score("NDCG (+UniProt)", results_ndcg_enriched_uniprot))
print(_fmt_row_score("NDCG (+chemical)", results_ndcg_enriched_chem))
print(_fmt_row_score("NDCG (+UniProt+chem)", results_ndcg_enriched_both))

## Optional: Save results

Save all evaluated conditions to JSON (e.g. `rag_reason_judge_rerank_eval_results_five_way.json`) for later comparison.

In [ ]:
# Save mean/std evaluation metrics for all conditions to a JSON file.
out_path = OUTPUT_DIR / "hybrid_bge_M3_rag_454_single_abs_chunk450_overlap20_seperate_pdb_id_bge_M3_reranker_large_0523.json"

def _metrics_entries(results_recall, results_precision, results_ndcg):
    return {
        str(k): {
            "recall_mean": results_recall[k][0], "recall_std": results_recall[k][1],
            "precision_mean": results_precision[k][0], "precision_std": results_precision[k][1],
            "ndcg_mean": results_ndcg[k][0], "ndcg_std": results_ndcg[k][1],
        } for k in K_LIST
    }

results_to_save = {
    "n_queries": len(dict_pmcid_abs),

    "retrieval_only": _metrics_entries(
        results_baseline, results_precision_baseline, results_ndcg_baseline
    ),
    "mrr_retrieval_only": mrr_baseline,
    "map_retrieval_only": map_baseline,

    "rerank_plain": _metrics_entries(
        results_rerank_plain, results_precision_plain, results_ndcg_plain
    ),
    "mrr_rerank_plain": mrr_plain,
    "map_rerank_plain": map_plain,

    "rerank_enriched_struct_only": _metrics_entries(
        results_rerank_enriched, results_precision_enriched, results_ndcg_enriched
    ),
    "mrr_rerank_enriched_struct_only": mrr_enriched,
    "map_rerank_enriched_struct_only": map_enriched,

    # "rerank_plain_rewritten": _metrics_entries(
    #     results_rerank_rewrite, results_precision_rewritten, results_ndcg_rewritten
    # ),
    # "mrr_rerank_plain_rewritteny": mrr_rewritten,
    # "map_rerank_plain_rewritten": map_rewritten,

    # "rerank_enriched_rewritten": _metrics_entries(
    #     results_rerank_enriched_rewrite, results_precision_enriched_rewritten, results_ndcg_enriched_rewritten
    # ),
    # "mrr_rerank_enriched_rewritteny": mrr_enriched_rewritten,
    # "map_rerank_enriched_rewritten": map_rewritten,

    "rerank_enriched_structure_uniprot": _metrics_entries(
        results_rerank_enriched_uniprot,
        results_precision_enriched_uniprot,
        results_ndcg_enriched_uniprot,
    ),
    "mrr_rerank_enriched_structure_uniprot": mrr_enriched_uniprot,
    "map_rerank_enriched_structure_uniprot": map_enriched_uniprot,

    "rerank_enriched_structure_chemical": _metrics_entries(
        results_rerank_enriched_chem,
        results_precision_enriched_chem,
        results_ndcg_enriched_chem,
    ),
    "mrr_rerank_enriched_structure_chemical": mrr_enriched_chem,
    "map_rerank_enriched_structure_chemical": map_enriched_chem,

    "rerank_enriched_structure_uniprot_and_chemical": _metrics_entries(
        results_rerank_enriched_both,
        results_precision_enriched_both,
        results_ndcg_enriched_both,
    ),
    "mrr_rerank_enriched_structure_uniprot_and_chemical": mrr_enriched_both,
    "map_rerank_enriched_structure_uniprot_and_chemical": map_enriched_both,

}

with open(out_path, "w") as f:
    json.dump(results_to_save, f, indent=2)
print(f"Results saved to {out_path}")


In [ ]:
# Save evaluation metrics with 95% CI bootstrap bounds to a JSON file.
out_path = OUTPUT_DIR / "95CI_hybrid_bge_M3_rag_454_single_abs_chunk450_overlap20_seperate_pdb_id_bge_M3_reranker_large_0522.json"


def _metrics_entries(results_recall, results_precision, results_ndcg):
    return {
        str(k): {
            "recall_mean": results_recall[k][0],
            "recall_ci_low": results_recall[k][1],
            "recall_ci_high": results_recall[k][2],
            "precision_mean": results_precision[k][0],
            "precision_ci_low": results_precision[k][1],
            "precision_ci_high": results_precision[k][2],
            "ndcg_mean": results_ndcg[k][0],
            "ndcg_ci_low": results_ndcg[k][1],
            "ndcg_ci_high": results_ndcg[k][2],
        }
        for k in K_LIST
    }


def _mrr_entry(mrr_tuple):
    return {
        "mean": mrr_tuple[0],
        "ci_low": mrr_tuple[1],
        "ci_high": mrr_tuple[2],
    }


results_to_save = {
    "n_queries": len(dict_pmcid_abs),
    "bootstrap": {"n_resamples": N_BOOT, "seed": BOOT_SEED, "ci": "95% percentile"},

    "retrieval_only": _metrics_entries(
        results_baseline, results_precision_baseline, results_ndcg_baseline
    ),
    "mrr_retrieval_only": _mrr_entry(mrr_baseline),
    "map_retrieval_only": _mrr_entry(map_baseline),

    "rerank_plain": _metrics_entries(
        results_rerank_plain, results_precision_plain, results_ndcg_plain
    ),
    "mrr_rerank_plain": _mrr_entry(mrr_plain),
    "map_rerank_plain": _mrr_entry(map_plain),

    "rerank_enriched_struct_only": _metrics_entries(
        results_rerank_enriched, results_precision_enriched, results_ndcg_enriched
    ),
    "mrr_rerank_enriched_struct_only": _mrr_entry(mrr_enriched),
    "map_rerank_enriched_struct_only": _mrr_entry(map_enriched),

    "rerank_enriched_structure_uniprot": _metrics_entries(
        results_rerank_enriched_uniprot,
        results_precision_enriched_uniprot,
        results_ndcg_enriched_uniprot,
    ),
    "mrr_rerank_enriched_structure_uniprot": _mrr_entry(mrr_enriched_uniprot),
    "map_rerank_enriched_structure_uniprot": _mrr_entry(map_enriched_uniprot),

    "rerank_enriched_structure_chemical": _metrics_entries(
        results_rerank_enriched_chem,
        results_precision_enriched_chem,
        results_ndcg_enriched_chem,
    ),
    "mrr_rerank_enriched_structure_chemical": _mrr_entry(mrr_enriched_chem),
    "map_rerank_enriched_structure_chemical": _mrr_entry(map_enriched_chem),

    "rerank_enriched_structure_uniprot_and_chemical": _metrics_entries(
        results_rerank_enriched_both,
        results_precision_enriched_both,
        results_ndcg_enriched_both,
    ),
    "mrr_rerank_enriched_structure_uniprot_and_chemical": _mrr_entry(mrr_enriched_both),
    "map_rerank_enriched_structure_uniprot_and_chemical": _mrr_entry(map_enriched_both),
}

with open(out_path, "w") as f:
    json.dump(results_to_save, f, indent=2)
print(f"Results saved to {out_path}")

In [ ]:
# (Empty placeholder cell.)


In [ ]:
# (Empty placeholder cell.)
